In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.preprocessing import StandardScaler

# 从CSV文件读取数据
data = pd.read_csv('E:/Desktop/Yield3.csv')

# 提取实验数据的输入变量和产量值
X = data[['LasR', 'ER', 'PR', 'XlnR', 'DHBR', 'RpaR', 'lacI', 'CinR']].values
y = data['Yield'].values

# 数据标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 定义高斯过程模型
kernel = C(1.0, (1e-6, 1e6)) * RBF(1.0, (1e-6, 1e2))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=50, alpha=1e-3)

# 拟合高斯过程回归模型
gp.fit(X_scaled, y)

# 打印内核参数
print(f"内核参数: {gp.kernel_}")

# 定义采样空间
bounds = np.array([
    [0.06, 15.2], [0.04, 14.1], [0.296, 23.56], [0.161, 40.75],
    [0.238, 11.9], [0.176, 31.6], [0.082, 6.02], [0.484, 13.26]
])
num_samples = 50000  # 减少采样点数
random_points = np.random.uniform(bounds[:, 0], bounds[:, 1], size=(num_samples, len(bounds)))

# 标准化随机点
random_points_scaled = scaler.transform(random_points)

# 对采样点分批预测均值
batch_size = 1000
mu_list = []
for i in range(0, random_points_scaled.shape[0], batch_size):
    batch = random_points_scaled[i:i + batch_size]
    mu_batch = gp.predict(batch, return_std=False)
    mu_list.append(mu_batch)

mu = np.concatenate(mu_list)

# 获取预测产量最高的64个点
top_indices = np.argsort(mu)[-64:]  # 获取均值最高的64个点的索引
top_points = random_points[top_indices]  # 获取原始坐标的点
top_mu = mu[top_indices]  # 对应的预测产量

# 将数据保存到DataFrame
output_data = pd.DataFrame(
    top_points, 
    columns=['LasR', 'ER', 'PR', 'XlnR', 'DHBR', 'RpaR', 'lacI', 'CinR']
)
output_data['Predicted_Yield'] = top_mu

# 保存到CSV文件
output_file = 'E:/Desktop/top_64_points.csv'
output_data.to_csv(output_file, index=False)
print(f"产量最高的64个点已保存到: {output_file}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.preprocessing import StandardScaler

# 从CSV文件读取数据
data = pd.read_csv('E:/Desktop/Yield3.csv')

# 提取实验数据的输入变量和产量值
X = data[['LasR', 'ER', 'PR', 'XlnR', 'DHBR', 'RpaR', 'lacI', 'CinR']].values
y = data['Yield'].values

# 数据标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 定义高斯过程模型
kernel = C(1.0, (1e-6, 1e6)) * RBF(1.0, (1e-6, 1e2))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=50, alpha=1e-3)

# 拟合高斯过程回归模型
gp.fit(X_scaled, y)

# 打印内核参数
print(f"内核参数: {gp.kernel_}")

# 定义采样空间
bounds = np.array([
    [0.06, 15.2], [0.04, 14.1], [0.296, 23.56], [0.161, 40.75],
    [0.238, 11.9], [0.176, 31.6], [0.082, 6.02], [0.484, 13.26]
])
num_samples = 50000  # 原始采样点数
random_points = np.random.uniform(bounds[:, 0], bounds[:, 1], size=(num_samples, len(bounds)))

# 标准化随机点
random_points_scaled = scaler.transform(random_points)

# 对采样点分批预测均值和标准差
batch_size = 1000
mu_list, sigma_list = [], []
for i in range(0, random_points_scaled.shape[0], batch_size):
    batch = random_points_scaled[i:i + batch_size]
    mu_batch, sigma_batch = gp.predict(batch, return_std=True)
    mu_list.append(mu_batch)
    sigma_list.append(sigma_batch)

mu = np.concatenate(mu_list)
sigma = np.concatenate(sigma_list)

# 缩小采样空间：选择置信区间小于阈值的点
confidence_threshold = 0.2  # 置信区间阈值（标准差）
filtered_indices = sigma < confidence_threshold
filtered_points = random_points[filtered_indices]
filtered_mu = mu[filtered_indices]

print(f"缩小后的采样空间点数: {len(filtered_points)}")

# 在缩小后的采样空间中选择预测产量最高的64个点
top_filtered_indices = np.argsort(filtered_mu)[-64:]
top_filtered_points = filtered_points[top_filtered_indices]
top_filtered_mu = filtered_mu[top_filtered_indices]

# 将缩小采样空间的结果保存到DataFrame
output_filtered_data = pd.DataFrame(
    top_filtered_points,
    columns=['LasR', 'ER', 'PR', 'XlnR', 'DHBR', 'RpaR', 'lacI', 'CinR']
)
output_filtered_data['Predicted_Yield'] = top_filtered_mu

# 保存到CSV文件
filtered_output_file = 'E:/Desktop/top_64_filtered_points.csv'
output_filtered_data.to_csv(filtered_output_file, index=False)
print(f"缩小采样空间后产量最高的64个点已保存到: {filtered_output_file}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.preprocessing import StandardScaler

# 1. 合并新实验数据
new_data = pd.read_csv('E:/Desktop/Yield4.csv')  # 新实验数据
data = pd.read_csv('E:/Desktop/Yield3.csv')  # 原始数据
combined_data = pd.concat([data, new_data], ignore_index=True)

# 提取输入变量和产量值
X = combined_data[['LasR', 'ER', 'PR', 'XlnR', 'DHBR', 'RpaR', 'lacI', 'CinR']].values
y = combined_data['Yield'].values

# 2. 数据标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. 重新定义和拟合高斯过程模型
kernel = C(1.0, (1e-6, 1e6)) * RBF(1.0, (1e-6, 1e2))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=50, alpha=1e-3)
gp.fit(X_scaled, y)

# 打印内核参数
print(f"更新后的内核参数: {gp.kernel_}")

# 4. 使用上一轮缩小的空间作为初始解空间
prev_filtered_points = pd.read_csv('E:/Desktop/top_64_filtered_points.csv').iloc[:, :-1].values  # 上一轮的输入点
num_samples = 20000  # 新采样点数
perturbation = 0.1  # 随机扰动幅度

# 在上一轮点附近生成新采样点
perturbed_points = np.array([
    prev_filtered_points[i] + np.random.uniform(-perturbation, perturbation, size=prev_filtered_points.shape[1])
    for i in range(len(prev_filtered_points))
])
perturbed_points = perturbed_points.reshape(-1, prev_filtered_points.shape[1])

# 确保点落在合理范围内
bounds = np.array([
    [0.06, 15.2], [0.04, 14.1], [0.296, 23.56], [0.161, 40.75],
    [0.238, 11.9], [0.176, 31.6], [0.082, 6.02], [0.484, 13.26]
])
perturbed_points = np.clip(perturbed_points, bounds[:, 0], bounds[:, 1])

# 标准化新采样点
perturbed_points_scaled = scaler.transform(perturbed_points)

# 对新采样点分批预测均值和标准差
batch_size = 1000
mu_list, sigma_list = [], []
for i in range(0, perturbed_points_scaled.shape[0], batch_size):
    batch = perturbed_points_scaled[i:i + batch_size]
    mu_batch, sigma_batch = gp.predict(batch, return_std=True)
    mu_list.append(mu_batch)
    sigma_list.append(sigma_batch)

mu = np.concatenate(mu_list)
sigma = np.concatenate(sigma_list)

# 5. 进一步缩小解空间
confidence_threshold = 0.15  # 更低的置信区间阈值
filtered_indices = sigma < confidence_threshold
filtered_points = perturbed_points[filtered_indices]
filtered_mu = mu[filtered_indices]

print(f"进一步缩小后的采样空间点数: {len(filtered_points)}")

# 在新的缩小空间中选择预测产量最高的64个点
top_filtered_indices = np.argsort(filtered_mu)[-64:]
top_filtered_points = filtered_points[top_filtered_indices]
top_filtered_mu = filtered_mu[top_filtered_indices]

# 将结果保存到DataFrame
output_filtered_data = pd.DataFrame(
    top_filtered_points,
    columns=['LasR', 'ER', 'PR', 'XlnR', 'DHBR', 'RpaR', 'lacI', 'CinR']
)
output_filtered_data['Predicted_Yield'] = top_filtered_mu

# 保存到CSV文件
next_filtered_output_file = 'E:/Desktop/next_top_64_filtered_points.csv'
output_filtered_data.to_csv(next_filtered_output_file, index=False)
print(f"下一轮实验点已保存到: {next_filtered_output_file}")


In [22]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
import matplotlib.patheffects as pe

# 创建保存图片的文件夹
save_dir = 'E:/Desktop/fig_optimalround3'
os.makedirs(save_dir, exist_ok=True)

# 加载训练数据
data1 = pd.read_csv('E:/Desktop/Yield3.csv')
data2 = pd.read_csv('E:/Desktop/round3.csv')
data3 = pd.read_csv('E:/Desktop/Yield5.csv')
data4 = pd.read_csv('E:/Desktop/Yield1.csv')
data5 = pd.read_csv('E:/Desktop/Yield0.csv')
data6 = pd.read_csv('E:/Desktop/Yield2.csv')
combined_data = pd.concat([data1, data2, data3, data4, data5, data6], ignore_index=True)
X = combined_data[['LasR', 'ER', 'PR', 'XlnR', 'DHBR', 'RpaR', 'lacI', 'CinR']].values
y = combined_data['Yield'].values

# 训练高斯过程回归模型
kernel = C(1.0, (1e-6, 1e6)) * RBF(length_scale=[1.0]*X.shape[1], length_scale_bounds=(1e-6, 1e4))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=50, alpha=1e-3)
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)
gp.fit(X_scaled, y)

# 加载每轮优化数据
first_round = pd.read_csv('E:/Desktop/Yield0.csv')
second_round = pd.read_csv('E:/Desktop/round2.csv')
third_round = pd.read_csv('E:/Desktop/round3.csv')

# 参数名及边界（已更新）
variables = ['LasR', 'ER', 'PR', 'XlnR', 'DHBR', 'RpaR', 'lacI', 'CinR']
bounds = np.array([
    [0.06, 15.2],     # LasR
    [0.04, 14.1],     # ER
    [0.296, 23.56],   # PR
    [0.161, 40.75],   # XlnR
    [0.238, 11.9],    # DHBR
    [0.176, 31.6],    # RpaR
    [0.082, 6.02],    # lacI
    [0.484, 13.26]    # CinR
])
fixed_values = combined_data[variables].mean(axis=0)

# 辅助函数：过滤超出边界的点
def filter_within_bounds(df, var1, var2, bounds, variables):
    i1 = variables.index(var1)
    i2 = variables.index(var2)
    return df[(df[var1] >= bounds[i1][0]) & (df[var1] <= bounds[i1][1]) &
              (df[var2] >= bounds[i2][0]) & (df[var2] <= bounds[i2][1])]

# 遍历所有变量对并绘图
for var1, var2 in combinations(variables, 2):
    plt.figure(figsize=(8, 6))
    ax = plt.gca()
    
    i1 = variables.index(var1)
    i2 = variables.index(var2)
    
    # 网格坐标
    x_grid = np.linspace(bounds[i1][0], bounds[i1][1], 50)
    y_grid = np.linspace(bounds[i2][0], bounds[i2][1], 50)
    xx, yy = np.meshgrid(x_grid, y_grid)
    
    # 预测点
    grid_samples = np.tile(fixed_values, (xx.size, 1))
    grid_samples[:, i1] = xx.ravel()
    grid_samples[:, i2] = yy.ravel()
    grid_scaled = scaler.transform(grid_samples)
    mu, _ = gp.predict(grid_scaled, return_std=True)



    mu = mu.reshape(xx.shape)

    # 热图
    contour = ax.contourf(xx, yy, mu, levels=20, cmap='viridis')
    plt.colorbar(contour, label='Predicted Yield', shrink=0.8)

    # 过滤每轮数据
    first_valid = filter_within_bounds(first_round, var1, var2, bounds, variables)
    second_valid = filter_within_bounds(second_round, var1, var2, bounds, variables)
    third_valid = filter_within_bounds(third_round, var1, var2, bounds, variables)

    # 散点图
    ax.scatter(first_valid[var1], first_valid[var2],
               c='red', s=80, edgecolor='k', label='Round 1 (n={})'.format(len(first_valid)))
    ax.scatter(second_valid[var1], second_valid[var2],
               c='blue', s=80, marker='s', edgecolor='k', label='Round 2 (n={})'.format(len(second_valid)))
    ax.scatter(third_valid[var1], third_valid[var2],
               c='white', s=80, marker='^', edgecolor='k', label='Round 3 (n={})'.format(len(third_valid)))

    # 均值路径
    first_mean = [first_valid[var1].mean(), first_valid[var2].mean()]
    second_mean = [second_valid[var1].mean(), second_valid[var2].mean()]
    third_mean = [third_valid[var1].mean(), third_valid[var2].mean()]
    
    ax.plot([first_mean[0], second_mean[0], third_mean[0]],
            [first_mean[1], second_mean[1], third_mean[1]],
            'w-', linewidth=2.5, alpha=0.8,
            path_effects=[pe.Stroke(linewidth=3.5, foreground='k'), pe.Normal()])
    
    # 注释优化路径变化
    annotation_text = (
        f"Optimization Progress:\n"
        f"Round1 → Round2: Δ={np.linalg.norm(np.array(second_mean) - np.array(first_mean)):.2f}\n"
        f"Round2 → Round3: Δ={np.linalg.norm(np.array(third_mean) - np.array(second_mean)):.2f}"
    )
    ax.annotate(annotation_text,
                xy=(0.05, 0.85), xycoords='axes fraction',
                fontsize=10, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", lw=0.5))

    # 轴标签和标题
    plt.title(f"{var1} vs {var2}")
    plt.xlabel(var1, fontsize=10)
    plt.ylabel(var2, fontsize=10)
    plt.legend()

    # 保存图像
    filename = f"{var1}_vs_{var2}.svg"
    save_path = os.path.join(save_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

print(f"所有图片已保存至：{save_dir}")


所有图片已保存至：E:/Desktop/fig_optimalround3
